# Data Loading and Library Import

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.metrics import (accuracy_score, f1_score, fbeta_score,
                             confusion_matrix, precision_recall_fscore_support)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
import xgboost as xgb

RANDOM_STATE = 67

In [2]:
df = pd.read_csv('filtered_data.csv')

In [3]:
df.head()

,_BMI5,_BMI5CAT,_AGE_G,_AGE80,_AGE65YR,_INCOMG1,INCOME3,_SMOKER3,_RFSMOK3,SMOKDAY2,...,MENTHLTH,ADDEPEV3,LANDSEX3,SEXVAR,DIABETE4,PERSDOC3,PRIMINS2,_URBSTAT,_IMPRACE,CHILDREN
0,2249.0,2.0,6.0,78.0,2.0,9.0,99.0,4.0,1.0,NaN,...,88.0,2.0,2.0,2.0,3.0,2.0,3.0,1.0,1.0,88.0
1,2583.0,3.0,6.0,80.0,2.0,7.0,11.0,3.0,1.0,3.0,...,88.0,2.0,1.0,1.0,3.0,1.0,3.0,1.0,1.0,88.0
2,2253.0,2.0,5.0,59.0,1.0,9.0,99.0,1.0,2.0,1.0,...,88.0,2.0,1.0,1.0,3.0,3.0,1.0,1.0,1.0,88.0
3,2509.0,3.0,6.0,80.0,2.0,4.0,6.0,4.0,1.0,NaN,...,88.0,2.0,1.0,1.0,3.0,1.0,3.0,1.0,1.0,88.0
4,1977.0,2.0,4.0,47.0,1.0,2.0,3.0,4.0,1.0,NaN,...,88.0,2.0,1.0,1.0,3.0,1.0,5.0,1.0,1.0,88.0


In [5]:
df.shape

(457670, 33)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 457670 entries, 0 to 457669
Data columns (total 33 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   _BMI5     414633 non-null  float64
 1   _BMI5CAT  414633 non-null  float64
 2   _AGE_G    457670 non-null  float64
 3   _AGE80    457670 non-null  float64
 4   _AGE65YR  457670 non-null  float64
 5   _INCOMG1  457670 non-null  float64
 6   INCOME3   448401 non-null  float64
 7   _SMOKER3  457670 non-null  float64
 8   _RFSMOK3  457670 non-null  float64
 9   SMOKDAY2  167076 non-null  float64
 10  CVDINFR4  457668 non-null  float64
 11  CVDCRHD4  457667 non-null  float64
 12  ASTHMA3   457667 non-null  float64
 13  _LTASTH1  457670 non-null  float64
 14  CHCKDNY2  457664 non-null  float64
 15  MARITAL   457661 non-null  float64
 16  EDUCA     457663 non-null  float64
 17  _EDUCAG   457670 non-null  float64
 18  GENHLTH   457665 non-null  float64
 19  EXERANY2  457667 non-null  float64
 20  _TOTINDA  45767

In [7]:
chosen_columns = ['_BMI5', '_AGE_G', 'INCOME3', '_SMOKER3', 'CVDINFR4', 'CVDCRHD4',
                 'ASTHMA3', 'CHCKDNY2', 'MARITAL', 'EDUCA', 'GENHLTH', 'EXERANY2',
                 'HAVARTH4', 'MENTHLTH', 'ADDEPEV3', 'SEXVAR', 'PERSDOC3', 'PRIMINS2',
                 '_URBSTAT', 'CHILDREN', 'DIABETE4']

df = df[chosen_columns]

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 457670 entries, 0 to 457669
Data columns (total 21 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   _BMI5     414633 non-null  float64
 1   _AGE_G    457670 non-null  float64
 2   INCOME3   448401 non-null  float64
 3   _SMOKER3  457670 non-null  float64
 4   CVDINFR4  457668 non-null  float64
 5   CVDCRHD4  457667 non-null  float64
 6   ASTHMA3   457667 non-null  float64
 7   CHCKDNY2  457664 non-null  float64
 8   MARITAL   457661 non-null  float64
 9   EDUCA     457663 non-null  float64
 10  GENHLTH   457665 non-null  float64
 11  EXERANY2  457667 non-null  float64
 12  HAVARTH4  457665 non-null  float64
 13  MENTHLTH  457667 non-null  float64
 14  ADDEPEV3  457665 non-null  float64
 15  SEXVAR    457670 non-null  float64
 16  PERSDOC3  457667 non-null  float64
 17  PRIMINS2  457667 non-null  float64
 18  _URBSTAT  443047 non-null  float64
 19  CHILDREN  452064 non-null  float64
 20  DIABETE4  45766

# Data Cleaning

From Modelling EDA we found out that there is an interesting situation here with some column having significant amount of "Don't know" or "Refused" answers. After few experiments from that same EDA, Random Forest Imputation with XGBoost-selected features yielded the best results overall, so we will be using it to replace the "Don't know" or "Refused" answers.

From data dictionary, we found out that there is some columns that are encoded weirdly, so we will be manually remapping those weird columns with a more proper values. Here are the list of changes that we made:

- Most of the binary column were mapped with 1 for yes, and 2 for no. To create a proper binary column, we've remapped them to the usual 0 for no and 1 for yes.
- Most categorical column uses 7/77 for don't know or not sure, 8/88 for 0, and 9/99 for refused, so we've swapped them with NaN values so that we can impute them with the previously stated reason above.
- Column GENHLTH's ordinality is reversed, where 1 was for excellent health while 5 was for poor health. In order to give the ordinality real meaning we've swapped the order of the value.